# KI und Algorithmen – Priorisierung

Dieses Notebook begleitet das Kapitel zu gewichteter Priorisierung und PageRank.
Es zeigt, wie aus mehreren Kriterien eine Rangliste entsteht und wie ein Netzwerk in eine Priorisierung übersetzt wird.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.precision', 3)

## 1) Gewichtete Priorisierung

Die Idee ist einfach: Jedes Kriterium bekommt ein Gewicht.
Der Score eines Objekts ist dann die gewichtete Summe der normierten Teilwerte.

In [ ]:
quellen = pd.DataFrame({
    'Quelle': ['A', 'B', 'C', 'D'],
    'Relevanz': [0.85, 0.75, 0.80, 0.70],
    'Aktualitaet': [0.50, 0.85, 0.65, 0.90],
    'Vertrauen': [0.75, 0.70, 0.95, 0.60],
    'Aufwand': [0.30, 0.20, 0.60, 0.10],
})

weights = {'Relevanz': 0.40, 'Aktualitaet': 0.25, 'Vertrauen': 0.25, 'Aufwand': 0.10}
quellen['Score'] = (
    weights['Relevanz'] * quellen['Relevanz'] +
    weights['Aktualitaet'] * quellen['Aktualitaet'] +
    weights['Vertrauen'] * quellen['Vertrauen'] +
    weights['Aufwand'] * (1 - quellen['Aufwand'])
)

ranking = quellen.sort_values('Score', ascending=False).reset_index(drop=True)
print(ranking[['Quelle', 'Score']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(ranking['Quelle'], ranking['Score'], color='#4C78A8')
ax.invert_yaxis()
ax.set_xlabel('Score')
ax.set_title('Gewichtete Priorisierung')
for index, value in enumerate(ranking['Score']):
    ax.text(value + 0.005, index, f'{value:.3f}', va='center')
plt.show()

## 2) PageRank

PageRank priorisiert Seiten in einem Netzwerk.
Eine Seite ist wichtig, wenn wichtige Seiten auf sie verlinken.
Der Dämpfungsfaktor modelliert den zufälligen Sprung zu einer beliebigen Seite.

In [ ]:
seiten = ['A', 'B', 'C', 'D']
links = np.array([
    [0, 1, 1, 0],
    [0, 0, 1, 0],
    [1, 0, 0, 0],
    [0, 0, 1, 0],
], dtype=float)

n = len(seiten)
d = 0.85
outdegree = links.sum(axis=1)
pagerank = np.ones(n) / n

for _ in range(100):
    neuer_pagerank = np.full(n, (1 - d) / n)
    for quelle in range(n):
        if outdegree[quelle] > 0:
            neuer_pagerank += d * pagerank[quelle] * links[quelle] / outdegree[quelle]
    if np.abs(neuer_pagerank - pagerank).sum() < 1e-12:
        pagerank = neuer_pagerank
        break
    pagerank = neuer_pagerank

pagerank_df = pd.DataFrame({'Seite': seiten, 'PageRank': pagerank}).sort_values('PageRank', ascending=False)
print(pagerank_df.to_string(index=False))
print('Summe:', round(pagerank.sum(), 6))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(pagerank_df['Seite'], pagerank_df['PageRank'], color='#F58518')
ax.set_xlabel('Seite')
ax.set_ylabel('PageRank')
ax.set_title('PageRank im kleinen Linknetz')
for index, value in enumerate(pagerank_df['PageRank']):
    ax.text(index, value + 0.005, f'{value:.3f}', ha='center')
plt.show()

## Übungen

1. Ändern Sie die Gewichte der gewichteten Priorisierung und beobachten Sie, wie sich das Ranking verändert.
2. Fügen Sie im PageRank-Beispiel eine neue Seite hinzu, die auf mehrere Seiten verlinkt, aber selbst keine eingehenden Links hat. Was passiert?
3. Wo wäre eine gewichtete Priorisierung im Schulalltag sinnvoll, wo eher PageRank?